In [ ]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA


data = pd.read_csv('data/application_train_FE_baked.csv')


# Correlation between features and target
correlations = data.corr()['TARGET'].abs().sort_values(ascending=False)
print(correlations)

TARGET                               1.000000
EXT_SOURCE_2                         0.160470
DAYS_BIRTH                           0.078161
prev_refused_rate                    0.076024
prev_share_mean_all                  0.073508
                                       ...   
NAME_INCOME_TYPE_Businessman         0.001692
FLAG_EMAIL                           0.001648
NAME_FAMILY_STATUS_Separated         0.001202
CODE_GENDER_XNA                      0.001070
NAME_HOUSING_TYPE_Co.op.apartment    0.000465
Name: TARGET, Length: 74, dtype: float64


In [ ]:
# New Features

def add_engineered_features(df, target="TARGET"):
    df = df.copy()
    eps = 1e-6

    # ---- Ratios / capacity ----
    df["credit_to_income"]  = df["AMT_CREDIT"]  / (df["AMT_INCOME_TOTAL"] + eps)
    df["annuity_to_income"] = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"] + eps)
    df["annuity_to_credit"] = df["AMT_ANNUITY"] / (df["AMT_CREDIT"] + eps)
    df["income_per_member"] = df["AMT_INCOME_TOTAL"] / np.maximum(df["CNT_FAM_MEMBERS"], 1)
    df["income_per_child"]  = df["AMT_INCOME_TOTAL"] / (1 + np.maximum(df["CNT_CHILDREN"], 0))
    df["dsr_monthly"]       = df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"]/12.0 + eps)

    # ---- Time conversions & contactability ----
    df["age_years"]    = -df["DAYS_BIRTH"] / 365.25
    df["reg_years"]    = -df["DAYS_REGISTRATION"] / 365.25
    df["id_years"]     = -df["DAYS_ID_PUBLISH"] / 365.25
    df["phone_years"]  = -df["DAYS_LAST_PHONE_CHANGE"] / 365.25
    df["contactability"] = df["FLAG_PHONE"] + df["FLAG_EMAIL"]

    # ---- Previous behavior & pricing ----
    df["prev_interest_per_credit"] = df["prev_interest_mean"] / (df["prev_amt_credit_mean"] + eps)
    df["prev_payments_ratio"]      = df["prev_last3_cnt_payment_mean"] / (df["prev_cnt_payment_mean"] + eps)
    df["recent_approval_momentum"] = df["prev_last3_approved_rate"] - df["prev_last5_approved_rate"]
    df["recent_credit_growth"]     = df["prev_last3_amt_credit_mean"] - df["prev_last5_amt_credit_mean"]
    df["prev_rate_spread"]         = df["prev_rate_mean_all"] - df["prev_rate_med_all"]

    # ---- Region transforms ----
    # If inputs may be standardized/negative, clip to keep log1p well-defined
    df["region_pop_log"]      = np.log1p(np.clip(df["REGION_POPULATION_RELATIVE"], 0, None))
    df["region_rating_x_pop"] = df["REGION_RATING"] * df["REGION_POPULATION_RELATIVE"]

    # ---- Magnitudes (log) & mild nonlinearity ----
    df["log_income"]  = np.log1p(np.clip(df["AMT_INCOME_TOTAL"], 0, None))
    df["log_credit"]  = np.log1p(np.clip(df["AMT_CREDIT"],       0, None))
    df["log_annuity"] = np.log1p(np.clip(df["AMT_ANNUITY"],      0, None))
    df["ext2_sq"]     = df["EXT_SOURCE_2"] ** 2

    # ---- Hygiene: replace inf -> NaN, then fill numerics (binary->0, continuous->median) ----
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_cols_no_t = [c for c in num_cols if c != target]

    bin_cols, cont_cols = [], []
    for c in num_cols_no_t:
        vals = pd.unique(df[c].dropna())
        if len(vals) <= 3 and set(vals).issubset({0, 1}):
            bin_cols.append(c)
        else:
            cont_cols.append(c)

    if bin_cols:
        df[bin_cols] = df[bin_cols].fillna(0)
    if cont_cols:
        df[cont_cols] = df[cont_cols].fillna(df[cont_cols].median())

    # Ensure target dtype if present; optional downcast for speed
    if target in df.columns:
        df[target] = df[target].astype(int)

    float_cols = [c for c in df.columns if c != target and pd.api.types.is_float_dtype(df[c])]
    df[float_cols] = df[float_cols].astype("float32")

    return df

data_fe = add_engineered_features(data)

In [58]:
# Make random split, stratified split by target variable, and stratified split by region rating, debt ratio, and target
train_data_random, val_data_random = train_test_split(data_fe, test_size=0.2, random_state=42)
train_data_stratified, val_data_stratified = train_test_split(data_fe, test_size=0.2, random_state=42, stratify=data_fe['TARGET'])

# # For custom stratification, create bins for continuous variables first
# data['REGION_RATING_bins'] = pd.qcut(data['REGION_RATING'], q=4, labels=['VL', 'L', 'H', 'VH'], duplicates='drop')
# data['debt_ratio_bins'] = pd.qcut(data['debt_ratio'], q=4, labels=['VL', 'L', 'H', 'VH'], duplicates='drop')

# # Create a combined stratification column
# data['strat_col'] = data['REGION_RATING_bins'].astype(str) + '_' + data['debt_ratio_bins'].astype(str) + '_' + data['TARGET'].astype(str)

# # Now stratify by the combined column
# train_data_custom, val_data_custom = train_test_split(data, test_size=0.2, random_state=42, stratify=data['strat_col'])

In [59]:
# Metrics
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def recall(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    possible_positives = np.sum(y_true == 1)
    return true_positives / possible_positives if possible_positives > 0 else 0

def precision(y_true, y_pred):
    true_positives = np.sum((y_true == 1) & (y_pred == 1))
    predicted_positives = np.sum(y_pred == 1)
    return true_positives / predicted_positives if predicted_positives > 0 else 0

def specificity(y_true, y_pred):
    true_negatives = np.sum((y_true == 0) & (y_pred == 0))
    possible_negatives = np.sum(y_true == 0)
    return true_negatives / possible_negatives if possible_negatives > 0 else 0

def f1_score(y_true, y_pred):
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    return 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0

def roc_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    P = np.sum(y_true == 1)
    N = np.sum(y_true == 0)
    if P == 0 or N == 0:
        return 0.5
    order = np.argsort(-y_score, kind='mergesort')
    y_sorted = y_true[order]
    s_sorted = y_score[order]
    tp = np.cumsum(y_sorted)
    fp = np.cumsum(1 - y_sorted)
    changes = np.where(np.diff(s_sorted) != 0)[0]
    cut_idx = np.r_[changes, len(s_sorted) - 1]
    tpr = tp[cut_idx] / P
    fpr = fp[cut_idx] / N
    tpr = np.r_[0.0, tpr, 1.0]
    fpr = np.r_[0.0, fpr, 1.0]

    return float(np.trapz(tpr, fpr))

In [60]:
def choose_threshold_at_recall(y_true, y_score, target_recall=0.80):
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(y_score, dtype=float)
    order = np.argsort(-s)
    y_sorted = y[order]
    s_sorted = s[order]
    P = int((y == 1).sum())
    N = int((y == 0).sum())
    if P == 0:
        return float(s_sorted.max() + 1e-12)
    tp = np.cumsum(y_sorted == 1)
    fp = np.cumsum(y_sorted == 0)
    tpr = tp / max(P, 1)
    fpr = fp / max(N, 1)
    ok = np.where(tpr >= target_recall)[0]
    if len(ok) == 0:
        return float(s_sorted.min() - 1e-12)

    i = ok[np.argmin(fpr[ok])]
    return float(s_sorted[i] - 1e-12)


def cross_validate(model, train_data, val_data, cv=5, target_col='TARGET', target_recall=0.80):

    def get_scores_proba_or_decision(m, X_tr, X_te):
        if hasattr(m, "predict_proba"):
            s_tr = m.predict_proba(X_tr)[:, 1]
            s_te = m.predict_proba(X_te)[:, 1]
            return s_tr, s_te
        elif hasattr(m, "decision_function"):
            s_tr_raw = m.decision_function(X_tr).astype(float)
            s_te_raw = m.decision_function(X_te).astype(float)
            mn, mx = s_tr_raw.min(), s_tr_raw.max()
            rng = (mx - mn) if (mx > mn) else 1.0
            s_tr = (s_tr_raw - mn) / rng
            s_te = (s_te_raw - mn) / rng
            return s_tr, s_te
        else:
            s_tr = m.predict(X_tr).astype(float)
            s_te = m.predict(X_te).astype(float)
            return s_tr, s_te

    X = train_data.drop(columns=[target_col]).to_numpy()
    y = train_data[target_col].to_numpy().astype(int)
    val_X = val_data.drop(columns=[target_col]).to_numpy()
    val_y = val_data[target_col].to_numpy().astype(int)
    n = len(X)
    fold_size = n // cv
    metrics = {'accuracy': [], 'recall': [], 'precision': [], 'specificity': [], 'f1_score': [], 'roc_auc': []}
    for fold in range(cv):
        start = fold * fold_size
        end = (fold + 1) * fold_size if fold != cv - 1 else n
        X_val_fold = X[start:end]
        y_val_fold = y[start:end]
        X_train_fold = np.concatenate([X[:start], X[end:]], axis=0)
        y_train_fold = np.concatenate([y[:start], y[end:]], axis=0)
        model.fit(X_train_fold, y_train_fold)
        s_tr, s_va = get_scores_proba_or_decision(model, X_train_fold, X_val_fold)
        thr = choose_threshold_at_recall(y_train_fold, s_tr, target_recall=target_recall)
        y_pred = (s_va >= thr).astype(int)

        metrics['accuracy'].append(accuracy(y_val_fold, y_pred))
        metrics['recall'].append(recall(y_val_fold, y_pred))
        metrics['precision'].append(precision(y_val_fold, y_pred))
        metrics['specificity'].append(specificity(y_val_fold, y_pred))
        metrics['f1_score'].append(f1_score(y_val_fold, y_pred))
        metrics['roc_auc'].append(roc_auc(y_val_fold, s_va))
    avg_metrics = {k: float(np.mean(v)) for k, v in metrics.items()}
    model.fit(X, y)
    s_tr_full, s_val = get_scores_proba_or_decision(model, X, val_X)
    thr_full = choose_threshold_at_recall(y, s_tr_full, target_recall=target_recall)
    y_val_pred = (s_val >= thr_full).astype(int)
    val_metrics = {
        'accuracy': accuracy(val_y, y_val_pred),
        'recall': recall(val_y, y_val_pred),
        'precision': precision(val_y, y_val_pred),
        'specificity': specificity(val_y, y_val_pred),
        'f1_score': f1_score(val_y, y_val_pred),
        'roc_auc': roc_auc(val_y, s_val),
        'threshold_used': thr_full
    }
    coefs = model.coef_ if hasattr(model, 'coef_') else None
    return avg_metrics, coefs, val_metrics

In [68]:
# Models
# Logistic Regression
log_reg = LogisticRegression(
    max_iter=3000, solver="lbfgs", penalty="l2", C=2.0,
    class_weight='balanced'
)

# Elastic-net LR
log_reg_en = LogisticRegression(
    max_iter=3000, solver="saga", penalty="elasticnet",
    l1_ratio=0.15, C=1.0, class_weight='balanced'
)

# Linear SVM 
base_svm = LinearSVC(
    class_weight='balanced', C=0.5, tol=5e-4, max_iter=8000, dual="auto"
)
svm_model = CalibratedClassifierCV(estimator=base_svm, method="sigmoid", cv=3)

# LDA
lda_model = LDA(solver="lsqr", shrinkage="auto")


In [69]:
# Run Models
# Define feature sets for different models
target = "TARGET"

# Basic demographic and financial features
predictors1 = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "debt_ratio",
    "CNT_CHILDREN", "CNT_FAM_MEMBERS", "REGION_POPULATION_RELATIVE",
    "REGION_RATING", "DAYS_BIRTH", "LIVE_CITY_NOT_WORK_CITY"
]

# Previous application statistics
predictors2 = [
    "prev_app_count", "prev_approved_rate", "prev_refused_rate",
    "prev_amt_credit_mean", "prev_cnt_payment_mean", "prev_ann_to_credit_mean",
    "prev_interest_mean", "prev_rate_mean_all", "prev_share_mean_all"
]

# Recent application history features
predictors3 = [
    "prev_last3_n", "prev_last3_approved_rate", "prev_last3_amt_credit_mean",
    "prev_last3_cnt_payment_mean", "prev_last3_ann_to_credit_mean",
    "prev_last5_n", "prev_last5_approved_rate", "prev_last5_cnt_payment_mean"
]

# Document and registration features
predictors4 = [
    "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE",
    "FLAG_PHONE", "FLAG_EMAIL", "LIVE_CITY_NOT_WORK_CITY",
    "REGION_POPULATION_RELATIVE", "REGION_RATING"
]

# Contract and property features
predictors5 = [
    "NAME_CONTRACT_TYPE_Cash.loans", "NAME_CONTRACT_TYPE_Revolving.loans",
    "FLAG_OWN_REALTY_Y", "FLAG_OWN_CAR_Y",
    "NAME_HOUSING_TYPE_House...apartment", "NAME_HOUSING_TYPE_Rented.apartment",
    "NAME_HOUSING_TYPE_With.parents", "AMT_CREDIT", "AMT_ANNUITY"
]

# Income and education features
predictors6 = [
    "NAME_INCOME_TYPE_Working", "NAME_INCOME_TYPE_Commercial.associate",
    "NAME_INCOME_TYPE_Pensioner", "NAME_INCOME_TYPE_State.servant",
    "NAME_EDUCATION_TYPE_Secondary...secondary.special",
    "NAME_EDUCATION_TYPE_Higher.education", "NAME_EDUCATION_TYPE_Lower.secondary",
    "AMT_INCOME_TOTAL", "REGION_RATING"
]

# Combined important features
predictors7 = [
    "EXT_SOURCE_2", "AMT_CREDIT", "AMT_ANNUITY", "AMT_INCOME_TOTAL",
    "debt_ratio", "prev_approved_rate", "prev_app_count",
    "DAYS_BIRTH", "DAYS_REGISTRATION", "REGION_RATING"
]

# Most important features subset
predictors8 = [
    "EXT_SOURCE_2", "AMT_CREDIT", "AMT_INCOME_TOTAL", "debt_ratio", "prev_approved_rate"
]

feature_sets = {
    'Model_1': predictors1,
    'Model_2': predictors2,
    'Model_3': predictors3,
    'Model_4': predictors4,
    'Model_5': predictors5,
    'Model_6': predictors6,
    'Model_7': predictors7,
    'Model_8': predictors8
}

# Model 1
train_df = train_data_random[feature_sets['Model_1'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_1'] + [target]].copy()
log_reg_metrics1 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics1

# Model 2
train_df = train_data_random[feature_sets['Model_2'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_2'] + [target]].copy()
log_reg_metrics2 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics2

# Model 3
train_df = train_data_random[feature_sets['Model_3'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_3'] + [target]].copy()
log_reg_metrics3 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics3

# Model 4
train_df = train_data_random[feature_sets['Model_4'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_4'] + [target]].copy()
log_reg_metrics4 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics4

# Model 5
train_df = train_data_random[feature_sets['Model_5'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_5'] + [target]].copy()
log_reg_metrics5 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics5

# Model 6
train_df = train_data_random[feature_sets['Model_6'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_6'] + [target]].copy()
log_reg_metrics6 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics6

# Model 7
train_df = train_data_random[feature_sets['Model_7'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_7'] + [target]].copy()
log_reg_metrics7 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics7

# Model 8
train_df = train_data_random[feature_sets['Model_8'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_8'] + [target]].copy()
log_reg_metrics8 = cross_validate(log_reg, train_df, val_df, cv=5)
log_reg_metrics8



({'accuracy': 0.4541284376012843,
  'recall': 0.7999986514166441,
  'precision': 0.10895130094394538,
  'specificity': 0.42366438225244074,
  'f1_score': 0.19177674946416906,
  'roc_auc': 0.6755321915826775},
 array([[-0.52100701, -0.03888583,  0.00180547,  0.31738746, -0.1946152 ]]),
 {'accuracy': 0.45214118107156825,
  'recall': 0.7932231067564809,
  'precision': 0.1064832575217844,
  'specificity': 0.42255042589739505,
  'f1_score': 0.18776121566448434,
  'roc_auc': 0.677031558203906,
  'threshold_used': 0.40143738087378766})

In [70]:
print(log_reg_metrics1)
print(log_reg_metrics2)
print(log_reg_metrics3)
print(log_reg_metrics4)
print(log_reg_metrics5)
print(log_reg_metrics6)
print(log_reg_metrics7)
print(log_reg_metrics8)

train_df = train_data_random[feature_sets['Model_8'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_8'] + [target]].copy()
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)

train_df = train_data_random[feature_sets['Model_8'] + [target]].copy()
val_df   = val_data_random[feature_sets['Model_8'] + [target]].copy()
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)

({'accuracy': 0.3872521078062424, 'recall': 0.7992925723905586, 'precision': 0.0978606349206339, 'specificity': 0.35096356329018435, 'f1_score': 0.17436613443942656, 'roc_auc': 0.6264747756897726}, array([[ 1.39344887e-04, -1.52286752e-01,  1.10590498e-01,
         3.01534474e-01,  7.38705127e-02, -1.12181103e-01,
        -2.74264548e-02,  2.02078487e-01,  2.66446183e-01,
         8.08449805e-02]]), {'accuracy': 0.38378307912918785, 'recall': 0.7999591753419065, 'precision': 0.09615998037050669, 'specificity': 0.34767748676264854, 'f1_score': 0.17168265650326367, 'roc_auc': 0.6197510236628124, 'threshold_used': 0.42980811825234033})
({'accuracy': 0.36085352719050307, 'recall': 0.7993451617874874, 'precision': 0.09411659358838691, 'specificity': 0.3222246725654295, 'f1_score': 0.16840020482652227, 'roc_auc': 0.6168166663845311}, array([[-0.09551471, -0.05294947,  0.23194448, -0.24135455,  0.05702806,
        -0.11690691,  0.0669896 ,  0.1868094 ,  0.07820111]]), {'accuracy': 0.358835223

In [64]:
predictors_AUC_7pp = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL","debt_ratio",
    "prev_approved_rate","prev_app_count","DAYS_BIRTH","DAYS_REGISTRATION","REGION_RATING",
    "credit_to_income","annuity_to_credit","age_years","reg_years",
    "prev_rate_spread","prev_interest_per_credit","ext2_sq"
]

# 2) Recent-behavior heavy (recency + momentum + a few capacity anchors)
predictors_RECENT = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_INCOME_TOTAL","debt_ratio","age_years",
    "prev_last3_approved_rate","prev_last5_approved_rate","recent_approval_momentum",
    "prev_last3_cnt_payment_mean","prev_last5_cnt_payment_mean",
    "prev_interest_per_credit","prev_rate_spread","ext2_sq"
]

# 3) Affordability & capacity focus (clean signal for linear models)
predictors_CAPACITY = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL","debt_ratio",
    "credit_to_income","annuity_to_income","annuity_to_credit","income_per_member","dsr_monthly",
    "REGION_RATING","age_years","prev_approved_rate","ext2_sq"
]

# 4) Stability + contactability + region interaction (often improves ranking)
predictors_STABILITY = [
    "EXT_SOURCE_2","REGION_RATING","REGION_POPULATION_RELATIVE","region_rating_x_pop",
    "DAYS_BIRTH","reg_years","id_years","phone_years","FLAG_PHONE","FLAG_EMAIL",
    "AMT_INCOME_TOTAL","debt_ratio","prev_approved_rate","AMT_CREDIT","ext2_sq"
]

# 5) Tiny, high-signal (9 vars) for speed + surprisingly good AUC
predictors_TINY9 = [
    "EXT_SOURCE_2","debt_ratio","prev_approved_rate",
    "credit_to_income","annuity_to_credit","age_years",
    "prev_rate_spread","prev_interest_per_credit","REGION_RATING"
]

# 6) Contract/property tilt + core finance (diversifies signal a bit)
predictors_CONTRACT = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","debt_ratio",
    "NAME_CONTRACT_TYPE_Cash.loans","NAME_CONTRACT_TYPE_Revolving.loans","FLAG_OWN_REALTY_Y",
    "REGION_RATING","credit_to_income","annuity_to_credit","prev_approved_rate","ext2_sq"
]

# Model AUC 7pp
train_df = train_data_random[predictors_AUC_7pp + [target]].copy()
val_df   = val_data_random[predictors_AUC_7pp + [target]].copy()
log_reg_metrics9 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics9)

# Model RECENT
train_df = train_data_random[predictors_RECENT + [target]].copy()
val_df   = val_data_random[predictors_RECENT + [target]].copy()
log_reg_metrics10 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics10)

# Model Capacity
train_df = train_data_random[predictors_CAPACITY + [target]].copy()
val_df   = val_data_random[predictors_CAPACITY + [target]].copy()
log_reg_metrics11 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics11)

# Model Stability
train_df = train_data_random[predictors_STABILITY + [target]].copy()
val_df   = val_data_random[predictors_STABILITY + [target]].copy()
log_reg_metrics12 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics12)

# Model Tiny9
train_df = train_data_random[predictors_TINY9 + [target]].copy()
val_df   = val_data_random[predictors_TINY9 + [target]].copy()
log_reg_metrics13 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics13)

# Model Contract
train_df = train_data_random[predictors_CONTRACT + [target]].copy()
val_df   = val_data_random[predictors_CONTRACT + [target]].copy()
log_reg_metrics14 = cross_validate(log_reg, train_df, val_df, cv=5)
print(log_reg_metrics14)


({'accuracy': 0.47444065534555346, 'recall': 0.8001569169495296, 'precision': 0.11282317654521298, 'specificity': 0.44575236157985315, 'f1_score': 0.19775520005934913, 'roc_auc': 0.6887309401150314}, array([[-5.07102311e-01, -9.84830070e-02,  1.10727233e-01,
        -3.88328609e-04,  2.92227659e-01, -2.19335540e-01,
        -3.63867465e-02,  2.23310274e-01,  3.42488602e-02,
         7.40606221e-02, -3.70909810e-04, -1.31556450e-04,
        -6.11390207e-04, -9.37682654e-05, -4.77026088e-03,
        -7.10480598e-06, -2.11078028e-02]]), {'accuracy': 0.47291748142354323, 'recall': 0.784037558685446, 'precision': 0.10933986165276552, 'specificity': 0.4459260833377605, 'f1_score': 0.19191565903867291, 'roc_auc': 0.6847862677617023, 'threshold_used': 0.4056267798011199})
({'accuracy': 0.4586667142784847, 'recall': 0.8000699201643796, 'precision': 0.10979764008014396, 'specificity': 0.42859035253751576, 'f1_score': 0.1930915855819541, 'roc_auc': 0.6795392214147601}, array([[-5.47822899e-01, -3

c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


({'accuracy': 0.46277721695907914, 'recall': 0.8000778747371886, 'precision': 0.11056736147204971, 'specificity': 0.43307074132451717, 'f1_score': 0.1942784675710479, 'roc_auc': 0.6823066109771075}, array([[-5.25876979e-01, -1.39709173e-01,  1.48227786e-01,
        -1.87453162e-01,  3.14015342e-01, -3.35063611e-04,
        -3.69807576e-02, -1.49365259e-04,  1.87980075e-01,
         3.01409172e-03,  7.43094228e-02, -2.39996407e+01,
        -1.92595527e-01, -2.63071304e-02]]), {'accuracy': 0.4633848259679312, 'recall': 0.792814860175546, 'precision': 0.10849162011173184, 'specificity': 0.4348049372221927, 'f1_score': 0.19086464040885526, 'roc_auc': 0.6827871687902246, 'threshold_used': 0.4047075238753094})
({'accuracy': 0.47248519303602243, 'recall': 0.7988039430954903, 'precision': 0.11229272088280647, 'specificity': 0.4437441543388774, 'f1_score': 0.19689893998396818, 'roc_auc': 0.6878843915171612}, array([[-0.49999965,  0.07230318,  0.01877466,  0.01626021,  0.2451566 ,
        -0.001

In [65]:
predictors_CONTRACT_2 = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","debt_ratio",
    "NAME_CONTRACT_TYPE_Cash.loans","NAME_CONTRACT_TYPE_Revolving.loans","FLAG_OWN_REALTY_Y",
    "REGION_RATING","credit_to_income","annuity_to_credit","prev_approved_rate","ext2_sq"
]

train_df = train_data_random[predictors_CONTRACT_2 + [target]].copy()
val_df   = val_data_random[predictors_CONTRACT_2 + [target]].copy()
train_strat_df = train_data_stratified[predictors_CONTRACT_2 + [target]].copy()
val_strat_df   = val_data_stratified[predictors_CONTRACT_2 + [target]].copy()

log_reg_metrics15 = cross_validate(log_reg, train_df, val_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)

log_reg_metrics15 = cross_validate(log_reg, train_strat_df, val_strat_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_strat_df, val_strat_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_strat_df, val_strat_df, cv=5)
print("LDA Metrics:", lda_metrics1)

Log Reg: ({'accuracy': 0.4603981142978439, 'recall': 0.7989426526421503, 'precision': 0.10999973022367424, 'specificity': 0.43057150207441, 'f1_score': 0.1933709436835646, 'roc_auc': 0.6802514198943806}, array([[-5.31556679e-01, -1.61731257e-01,  1.31590013e-01,
         3.22795578e-01,  1.11422636e-01, -3.00827823e-01,
        -4.14161359e-02,  6.94415421e-02, -3.47558414e-04,
        -1.43763302e-04, -1.86383180e-01, -2.84235241e-02]]), {'accuracy': 0.4588384825967931, 'recall': 0.7993468054705042, 'precision': 0.10834739783637219, 'specificity': 0.42929749065859146, 'f1_score': 0.1908289069733444, 'roc_auc': 0.6818666421179391, 'threshold_used': 0.40384689569915394})
SVM Metrics: ({'accuracy': 0.4614980500225533, 'recall': 0.7992477428367063, 'precision': 0.11023877950428385, 'specificity': 0.43174172878019046, 'f1_score': 0.19374910564602657, 'roc_auc': 0.6801951170387015}, array([[-2.49816635e-01, -7.30807689e-02,  5.98201945e-02,
         1.44827327e-01,  4.98374558e-02, -1.34708

In [54]:
predictors_COEF_CORE20 = [
    "EXT_SOURCE_2","debt_ratio","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL",
    "credit_to_income","annuity_to_credit","annuity_to_income",
    "age_years","reg_years","REGION_RATING",
    "prev_approved_rate","prev_app_count",
    "prev_rate_spread","prev_interest_per_credit","prev_cnt_payment_mean",
    "prev_last3_approved_rate","prev_last5_approved_rate",
    "ext2_sq","region_rating_x_pop"
]

predictors_COEF_RECENT_CAP = [
    "EXT_SOURCE_2","debt_ratio","credit_to_income","annuity_to_credit","dsr_monthly","age_years",
    "prev_last3_approved_rate","prev_last5_approved_rate","recent_approval_momentum",
    "prev_last3_cnt_payment_mean","prev_last5_cnt_payment_mean",
    "prev_interest_per_credit","prev_rate_spread","recent_credit_growth",
    "prev_payments_ratio","REGION_RATING","ext2_sq"
]

predictors_COEF_TINY12 = [
    "EXT_SOURCE_2","debt_ratio","prev_approved_rate",
    "credit_to_income","annuity_to_credit","age_years",
    "prev_rate_spread","prev_interest_per_credit","REGION_RATING",
    "prev_last3_approved_rate","prev_app_count","ext2_sq"
]

predictors_COEF_NOISE_REDUCED = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","debt_ratio",
    "credit_to_income","annuity_to_credit","income_per_member",
    "age_years","REGION_RATING","prev_approved_rate","prev_app_count",
    "prev_last3_approved_rate","prev_rate_spread","prev_interest_per_credit",
    "region_rating_x_pop","ext2_sq"
]


feature_sets_new = {
    "COEF_CORE20": predictors_COEF_CORE20,
    "COEF_RECENT_CAP": predictors_COEF_RECENT_CAP,
    "COEF_TINY12": predictors_COEF_TINY12,
    "COEF_NOISE_REDUCED": predictors_COEF_NOISE_REDUCED,
}

for name, cols in feature_sets_new.items():
    train_df = train_data_random[cols + [target]].copy()
    val_df   = val_data_random[cols + [target]].copy()
    print(f"\n=== {name} ===")
    print("LR:",  cross_validate(log_reg, train_df, val_df, cv=5))
    print("SVM:", cross_validate(svm_model, train_df, val_df, cv=5))
    print("LDA:", cross_validate(lda_model, train_df, val_df, cv=5))


KeyError: "['age_years', 'reg_years', 'region_rating_x_pop'] not in index"

In [76]:
# get all columns except ID
all_features = [col for col in data_fe.columns if col != 'SK_ID_CURR']
train_all = train_data_random[all_features].copy()
val_all = val_data_random[all_features].copy()

log_reg_metrics_all = cross_validate(log_reg, train_all, val_all, cv=5)
print(log_reg_metrics_all)
svc_metrics_all = cross_validate(svm_model, train_all, val_all, cv=5)
print(svc_metrics_all)
lda_metrics_all = cross_validate(lda_model, train_all, val_all, cv=5)
print(lda_metrics_all)


KeyError: "['income_per_member', 'income_per_child', 'age_years', 'reg_years', 'id_years', 'phone_years', 'contactability', 'region_pop_log', 'region_rating_x_pop', 'log_income', 'log_credit', 'log_annuity'] not in index"

Trying new feature engineering















In [ ]:
def add_legal_features(df, target="TARGET"):
    """Engineered features that avoid protected/demographic/geographic attributes."""
    out = df.copy()
    eps = 1e-6

    # -------- capacity / affordability (no age) --------
    # (works on standardized data; still useful as transforms)
    out["credit_to_income"]  = out["AMT_CREDIT"]  / (np.abs(out["AMT_INCOME_TOTAL"]) + eps)
    out["annuity_to_income"] = out["AMT_ANNUITY"] / (np.abs(out["AMT_INCOME_TOTAL"]) + eps)
    out["annuity_to_credit"] = out["AMT_ANNUITY"] / (np.abs(out["AMT_CREDIT"]) + eps)
    out["dsr_monthly"]       = out["AMT_ANNUITY"] / (np.abs(out["AMT_INCOME_TOTAL"])/12.0 + eps)

    # softplus to make safe positive transforms on standardized variables
    softplus = lambda x: np.log1p(np.exp(np.clip(x, -20, 20)))
    out["sp_income"]  = softplus(out["AMT_INCOME_TOTAL"])
    out["sp_credit"]  = softplus(out["AMT_CREDIT"])
    out["sp_annuity"] = softplus(out["AMT_ANNUITY"])

    # -------- prior behavior / pricing (legal) --------
    out["prev_interest_per_credit"] = out["prev_interest_mean"] / (np.abs(out["prev_amt_credit_mean"]) + eps)
    out["prev_payments_ratio"]      = out["prev_last3_cnt_payment_mean"] / (np.abs(out["prev_cnt_payment_mean"]) + eps)
    out["recent_approval_momentum"] = out["prev_last3_approved_rate"] - out["prev_last5_approved_rate"]
    out["recent_credit_growth"]     = out["prev_last3_amt_credit_mean"] - out["prev_last5_amt_credit_mean"]
    out["prev_rate_spread"]         = out["prev_rate_mean_all"] - out["prev_rate_med_all"]

    # -------- nonlinear bumps (bounded / stable) --------
    out["ext2_sq"]           = out["EXT_SOURCE_2"] ** 2
    out["debt_ratio_sq"]     = out["debt_ratio"] ** 2
    out["cti_sq"]            = out["credit_to_income"] ** 2
    out["a2c_sq"]            = out["annuity_to_credit"] ** 2

    # -------- interactions anchored on model-strong signals --------
    out["ext2_x_dsr"]        = out["EXT_SOURCE_2"] * out["dsr_monthly"]
    out["ext2_x_cti"]        = out["EXT_SOURCE_2"] * out["credit_to_income"]
    out["ext2_x_approved"]   = out["EXT_SOURCE_2"] * out["prev_approved_rate"]
    out["pricing_x_capacity"]= out["prev_rate_spread"] * out["credit_to_income"]
    out["payhist_x_approved"]= out["prev_cnt_payment_mean"] * out["prev_approved_rate"]
    out["interest_x_burden"] = out["prev_interest_per_credit"] * out["annuity_to_credit"]
    out["recent_x_pay"]      = out["recent_approval_momentum"] * out["prev_last3_cnt_payment_mean"]
    out["approved_x_count"]  = out["prev_approved_rate"] * out["prev_app_count"]

    # hygiene
    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    num_cols = [c for c in out.columns if c != target and pd.api.types.is_numeric_dtype(out[c])]
    out[num_cols] = out[num_cols].fillna(out[num_cols].median())

    return out

# Build the engineered (legal) frame
data_legal = add_legal_features(data)


train_data_random_2, val_data_random_2 = train_test_split(data_legal, test_size=0.2, random_state=42)
train_data_stratified_2, val_data_stratified_2 = train_test_split(data_legal, test_size=0.2, random_state=42, stratify=data_legal['TARGET'])


target = "TARGET"

predictors_LEGAL_CORE20 = [
    "EXT_SOURCE_2","debt_ratio","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL",
    "credit_to_income","annuity_to_income","annuity_to_credit","dsr_monthly",
    "prev_approved_rate","prev_app_count","prev_cnt_payment_mean",
    "prev_rate_spread","prev_interest_per_credit","recent_approval_momentum",
    "recent_credit_growth","prev_payments_ratio","ext2_sq","debt_ratio_sq","pricing_x_capacity"
]

predictors_LEGAL_RECENT = [
    "EXT_SOURCE_2","debt_ratio","AMT_CREDIT","AMT_INCOME_TOTAL",
    "prev_last3_approved_rate","prev_last5_approved_rate","recent_approval_momentum",
    "prev_last3_cnt_payment_mean","prev_last5_cnt_payment_mean",
    "prev_interest_per_credit","prev_rate_spread",
    "ext2_sq","ext2_x_dsr","recent_x_pay","approved_x_count"
]

predictors_LEGAL_CAPACITY = [
    "EXT_SOURCE_2","AMT_CREDIT","AMT_ANNUITY","AMT_INCOME_TOTAL","debt_ratio",
    "credit_to_income","annuity_to_income","annuity_to_credit","dsr_monthly",
    "sp_income","sp_credit","sp_annuity",
    "prev_approved_rate","prev_rate_spread","prev_interest_per_credit","ext2_sq","a2c_sq","cti_sq"
]

predictors_LEGAL_TINY12 = [
    "EXT_SOURCE_2","debt_ratio","prev_approved_rate",
    "credit_to_income","annuity_to_credit","prev_rate_spread",
    "prev_interest_per_credit","prev_cnt_payment_mean",
    "ext2_sq","pricing_x_capacity","approved_x_count","dsr_monthly"
]


train_df = train_data_random_2[predictors_LEGAL_CORE20 + [target]].copy()
val_df   = val_data_random_2[predictors_LEGAL_CORE20 + [target]].copy()
train_strat_df = train_data_stratified_2[predictors_LEGAL_CORE20 + [target]].copy()
val_strat_df   = val_data_stratified_2[predictors_LEGAL_CORE20 + [target]].copy()

log_reg_metrics15 = cross_validate(log_reg, train_df, val_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)
log_reg_metrics15 = cross_validate(log_reg, train_strat_df, val_strat_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_strat_df, val_strat_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_strat_df, val_strat_df, cv=5)
print("LDA Metrics:", lda_metrics1)


In [ ]:
train_df = train_data_random_2[predictors_LEGAL_RECENT + [target]].copy()
val_df   = val_data_random_2[predictors_LEGAL_RECENT + [target]].copy()
train_strat_df = train_data_stratified_2[predictors_LEGAL_RECENT + [target]].copy()
val_strat_df   = val_data_stratified_2[predictors_LEGAL_RECENT + [target]].copy()

log_reg_metrics15 = cross_validate(log_reg, train_df, val_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)
log_reg_metrics15 = cross_validate(log_reg, train_strat_df, val_strat_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_strat_df, val_strat_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_strat_df, val_strat_df, cv=5)
print("LDA Metrics:", lda_metrics1)



Log Reg: ({'accuracy': 0.4534929267889036, 'recall': 0.8000227757976088, 'precision': 0.10883500415400445, 'specificity': 0.42296578637742516, 'f1_score': 0.19159902866936873, 'roc_auc': 0.6752337875901052}, array([[-5.56103695e-01,  3.15246631e-01, -3.97095112e-02,
         2.10903245e-03, -4.29529849e-02, -1.13459118e-01,
         7.05061333e-02, -3.07150599e-02,  4.62740664e-02,
        -3.77175587e-06, -5.61717568e-02, -3.39228019e-02,
         2.56784756e-05, -7.89454681e-02, -4.19218054e-02]]), {'accuracy': 0.4507886846564985, 'recall': 0.7952643396611553, 'precision': 0.10645681340000547, 'specificity': 0.4209035045777329, 'f1_score': 0.18777713514555622, 'roc_auc': 0.677822677382928, 'threshold_used': 0.4011446808945526})
SVM Metrics: ({'accuracy': 0.4533462633816672, 'recall': 0.7997907662985659, 'precision': 0.10878292438104338, 'specificity': 0.4228293277267536, 'f1_score': 0.19151102776246462, 'roc_auc': 0.675179416646512}, None, {'accuracy': 0.4514567852952679, 'recall': 0

In [ ]:

train_df = train_data_random_2[predictors_LEGAL_CAPACITY + [target]].copy()
val_df   = val_data_random_2[predictors_LEGAL_CAPACITY + [target]].copy()
train_strat_df = train_data_stratified_2[predictors_LEGAL_CAPACITY + [target]].copy()
val_strat_df   = val_data_stratified_2[predictors_LEGAL_CAPACITY + [target]].copy()

log_reg_metrics15 = cross_validate(log_reg, train_df, val_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)
log_reg_metrics15 = cross_validate(log_reg, train_strat_df, val_strat_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_strat_df, val_strat_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_strat_df, val_strat_df, cv=5)
print("LDA Metrics:", lda_metrics1)



c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

Log Reg: ({'accuracy': 0.44913390248297536, 'recall': 0.7993619473713974, 'precision': 0.10797573721232265, 'specificity': 0.4182881404756424, 'f1_score': 0.19024552184686414, 'roc_auc': 0.675498892090773}, array([[-5.12874027e-01, -6.12355085e-03,  1.22117679e-01,
         2.17135300e-02,  3.01666916e-01, -9.04559757e-04,
        -2.44142861e-02,  1.81448648e-03,  2.13367333e-03,
        -5.58349871e-02, -9.77136294e-02, -3.33018344e-02,
        -2.47344080e-01, -5.52320394e-03, -7.07034390e-06,
         3.48999896e-02, -4.32784009e-07,  1.07207415e-07]]), {'accuracy': 0.4499902229174814, 'recall': 0.7924066135946112, 'precision': 0.10601343601507456, 'specificity': 0.42028369547893535, 'f1_score': 0.18700773177252694, 'roc_auc': 0.6760361494543097, 'threshold_used': 0.3928613453889574})


c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


SVM Metrics: ({'accuracy': 0.461241375368477, 'recall': 0.7995450871811891, 'precision': 0.11022262331383684, 'specificity': 0.43144292583405813, 'f1_score': 0.19373142978121588, 'roc_auc': 0.6798659156487762}, None, {'accuracy': 0.46169013166471123, 'recall': 0.7952643396611553, 'precision': 0.10843910042306835, 'specificity': 0.4327507127804636, 'f1_score': 0.19085409165503223, 'roc_auc': 0.6811914906617941, 'threshold_used': 0.056615453070863256})
LDA Metrics: ({'accuracy': 0.4480869142719521, 'recall': 0.7996399434895345, 'precision': 0.10781800179708792, 'specificity': 0.4171206243003008, 'f1_score': 0.1900100992186003, 'roc_auc': 0.6750755181232965}, array([[-4.91218412e-01, -3.78151433e-03,  1.68345048e-01,
         2.25062213e-02,  3.56502862e-01,  1.33642768e-04,
         4.40870051e-05,  1.10659264e-03,  7.48897174e-06,
        -1.18826611e-01, -1.96361115e-01, -1.05075437e-01,
        -2.10498197e-01, -4.20106752e-02, -1.85626101e-06,
         1.52956293e-01, -1.38066402e-07

c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\ellio\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/st

Log Reg: ({'accuracy': 0.45018901683778545, 'recall': 0.7990029397442446, 'precision': 0.10785802049450825, 'specificity': 0.41955976184963684, 'f1_score': 0.19005296645090466, 'roc_auc': 0.6752050880243088}, array([[-4.59588446e-01,  1.02028105e-02,  1.04849994e-01,
         2.60207810e-02,  2.69902002e-01, -1.18248597e-04,
        -1.28650803e-02,  3.34043197e-04,  1.13810671e-03,
        -6.05170635e-02, -8.75116522e-02, -4.32930877e-02,
        -2.36980966e-01, -5.60404366e-03,  9.74473412e-06,
         3.37885738e-02, -1.08434496e-08, -5.87878518e-08]]), {'accuracy': 0.44777408421327075, 'recall': 0.8042381432896064, 'precision': 0.10798287448515066, 'specificity': 0.41646429014588837, 'f1_score': 0.19040110848324135, 'roc_auc': 0.6827519252518586, 'threshold_used': 0.3936498823923538})
SVM Metrics: ({'accuracy': 0.4575708128737023, 'recall': 0.7998057030025627, 'precision': 0.10928836143939584, 'specificity': 0.42751672522252304, 'f1_score': 0.1922952309816646, 'roc_auc': 0.67827

In [ ]:
train_df = train_data_random_2[predictors_LEGAL_TINY12 + [target]].copy()
val_df   = val_data_random_2[predictors_LEGAL_TINY12 + [target]].copy()
train_strat_df = train_data_stratified_2[predictors_LEGAL_TINY12 + [target]].copy()
val_strat_df   = val_data_stratified_2[predictors_LEGAL_TINY12 + [target]].copy()

log_reg_metrics15 = cross_validate(log_reg, train_df, val_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_df, val_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_df, val_df, cv=5)
print("LDA Metrics:", lda_metrics1)
log_reg_metrics15 = cross_validate(log_reg, train_strat_df, val_strat_df, cv=5)
print("Log Reg:", log_reg_metrics15)
svc_metrics1 = cross_validate(svm_model, train_strat_df, val_strat_df, cv=5)
print("SVM Metrics:", svc_metrics1)
lda_metrics1 = cross_validate(lda_model, train_strat_df, val_strat_df, cv=5)
print("LDA Metrics:", lda_metrics1)

Log Reg: ({'accuracy': 0.45656868586214217, 'recall': 0.7997112040204415, 'precision': 0.10937170775159744, 'specificity': 0.4263393281276947, 'f1_score': 0.1924216889136608, 'roc_auc': 0.6770340135454617}, array([[-5.62746848e-01,  3.14011805e-01, -1.74015430e-01,
        -7.64790733e-04,  1.46778058e-03, -4.32868102e-02,
        -3.92898553e-06,  3.25219908e-02, -3.55982802e-02,
        -1.84239522e-03, -4.37491483e-02,  8.49917792e-05]]), {'accuracy': 0.4553513231651675, 'recall': 0.7936313533374157, 'precision': 0.10710448747968375, 'specificity': 0.4260036480192672, 'f1_score': 0.1887378640776699, 'roc_auc': 0.6788294817420303, 'threshold_used': 0.40143453704533066})
SVM Metrics: ({'accuracy': 0.4562549947725191, 'recall': 0.7997806832006822, 'precision': 0.1093202986171186, 'specificity': 0.42599395493551906, 'f1_score': 0.19234376435201506, 'roc_auc': 0.6769097229907295}, None, {'accuracy': 0.45541650371529135, 'recall': 0.7924066135946112, 'precision': 0.1069863579991732, 'spec

In [ ]:
# get all columns except ID
all_features = [col for col in data_legal.columns if col != 'SK_ID_CURR']
train_all = train_data_random_2[all_features].copy()
val_all = val_data_random_2[all_features].copy()

log_reg_metrics_all = cross_validate(log_reg, train_all, val_all, cv=5)
print(log_reg_metrics_all)
svc_metrics_all = cross_validate(svm_model, train_all, val_all, cv=5)
print(svc_metrics_all)
lda_metrics_all = cross_validate(lda_model, train_all, val_all, cv=5)
print(lda_metrics_all)

NameError: name 'train_data_random_2' is not defined